In [ ]:
import os 
import sys
import numpy as np
import pandas as pd
from xspec import *


# pha = "spec.pi"

ch = Xset.chatter
Xset.chatter = 20
lch = Xset.logChatter
Xset.logChatter = 20
logFile = Xset.openLog("xspec.log")
logFile = Xset.log

s1 = Spectrum("spec.pi")
s1.response = "rspdata/swxpc0to12s6_20010101v011.rmf"
s1.response.arf = "spec.arf"
s1.background = "spec_bkg.pi"
AllData.ignore("bad")
s1.ignore("**-0.3,10.0-**")

m1 = Model("tbabs*powerlaw")
m1.TBabs.nH = 0.0237
m1.TBabs.nH.frozen = True
m1.powerlaw.PhoIndex = 1.0
Fit.nIterations = 100
Fit.statMethod = "chi"
Fit.perform()
Fit.show()

AllModels.calcFlux("0.3 10.0 err")

Plot.device = f"plot.ps"
Plot.xAxis = "Hz"
Plot.xLog = True
Plot.yLog = True
Plot.setRebin(20, 20)
Plot("eeufspec","ratio")

Plot.device = "/null"
Plot("eeufspec")
xVals = Plot.x()
yVals = Plot.y()
modVals = Plot.model()
yErrs = Plot.yErr()
dataM = np.column_stack((xVals, yVals, yErrs, modVals))
np.savetxt("spec.csv", dataM, delimiter=",", header="xVals,yVals,yErrs,modVals", comments="")
Plot("ratio")
xVals = Plot.x()
yVals = Plot.y()
yErrs = Plot.yErr()
dataR = np.column_stack((xVals, yVals, yErrs))
np.savetxt("ratio.csv", dataR, delimiter=",", header="xVals,yVals,yErrs", comments="")

AllData.clear()
Xset.closeLog()

In [ ]:
import os 
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, LogLocator

phapath = os.getcwd()

scsv = os.path.join(phapath, "spec.csv")
df_spect = pd.read_csv(scsv)
x1, y1, y1e, mdl = df_spect["xVals"], df_spect["yVals"], df_spect["yErrs"], df_spect["modVals"]

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(x1, y1, yerr=y1e, fmt="o", markersize=4, label="Data", color="royalblue", ecolor="lightblue", elinewidth=1, capsize=1)
ax.plot(x1, mdl, label="Model", color="crimson", linestyle="-", linewidth=1)
ax.legend()
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1e10, 1e30)
ax.set_ylim(1e-13, 1e-8) 
ax.set_xlabel('ν (Hz)')
ax.set_ylabel(r'νFν (Erg cm$^{-2}$ s$^{-1}$)')
ax.tick_params(axis="both", direction="in", length=3, width=0.5, colors="black", pad=4, which="both", labelsize=8)
plt.tight_layout()
plt.savefig(f"{phapath}/spec.png", dpi=300, bbox_inches="tight")
plt.show()